<a href="https://colab.research.google.com/github/JabulaniMcineka/MyProjects/blob/main/Clinic_data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving final_sample_20250924_allhhmebers_2.xlsx to final_sample_20250924_allhhmebers_2.xlsx
Saving S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701.csv to S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701 (1).csv


In [ ]:
!ls



'All_clinics 1.csv'
 final_sample_20250924_allhhmebers_2.xlsx
'S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701 (1).csv'
 S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701.csv
 sample_data


In [ ]:
import pandas as pd

# Load clinics file
import pandas as pd

clinics_df = pd.read_csv("All_clinics 1.csv", sep=';')
clinics_df.head()


# Load eligible individuals file
eligible_df = pd.read_csv("S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701.csv")


# Load Excel file
household_df = pd.read_excel("final_sample_20250924_allhhmebers_2.xlsx")

household_df.head()


print("Files loaded successfully ")


Files loaded successfully 


In [ ]:
import pandas as pd

# Load clinics file
clinics_df = pd.read_csv("All_clinics 1.csv", sep=None, engine='python')

# Load eligible individuals file
eligible_df = pd.read_csv(
    "S3E-S3EEligibleIndividua_DATA_LABELS_2026-02-19_0701.csv",
    sep=None,
    engine='python'
)

# Load household file (Excel does NOT use sep)
household_df = pd.read_excel("final_sample_20250924_allhhmebers_2.xlsx")

print("Files loaded successfully")

print("Clinics shape:", clinics_df.shape)
print("Eligible shape:", eligible_df.shape)
print("Household shape:", household_df.shape)

print("Eligible shape:", eligible_df.shape)


Files loaded successfully
Clinics shape: (6275, 15)
Eligible shape: (6295, 5)
Household shape: (89663, 58)
Eligible shape: (6295, 5)


In [ ]:
# Create a clean copy with selected columns
eligible_full_df = eligible_flood_df[[
    'Record ID',
    'Event Name',
    "Participant's Gender",
    'incl_ctrl_age',
    'Bounded Structure ID Number',
    'flood_experience'
]].copy()

# Now it's safe to modify
# Better anonymization that preserves relationships
record_ids = [f"RID_{i:06d}" for i in range(1, len(eligible_full_df) + 1)]
eligible_full_df['Record ID'] = record_ids

# For structure IDs, consider preserving uniqueness
unique_structures = eligible_full_df['Bounded Structure ID Number'].unique()
struct_map = {old: f"Struct_{i+1}" for i, old in enumerate(unique_structures)}
eligible_full_df['Bounded Structure ID Number'] = eligible_full_df['Bounded Structure ID Number'].map(struct_map)

# Optional: Add metadata about the transformation
eligible_full_df.attrs['created_from'] = 'eligible_flood_df'
eligible_full_df.attrs['creation_date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')


In [114]:
import pandas as pd

# --- Step 0: Ensure IDs are numeric ---
eligible_df['Bounded Structure ID Number'] = pd.to_numeric(
    eligible_df['Bounded Structure ID Number'], errors='coerce'
).astype('Int64')

clinics_df['incl_ctrl_bsid'] = pd.to_numeric(
    clinics_df['incl_ctrl_bsid'], errors='coerce'
).astype('Int64')

household_df['bs_id'] = pd.to_numeric(
    household_df['bs_id'], errors='coerce'
).astype('Int64')

# --- Step 1: Remove duplicates from clinics_df (keep first occurrence) ---
clinics_df_deduplicated = clinics_df.drop_duplicates(subset=['incl_ctrl_bsid'], keep='first')

print(f"Original clinics rows: {len(clinics_df)}")
print(f"After removing duplicates: {len(clinics_df_deduplicated)}")
print(f"Removed {len(clinics_df) - len(clinics_df_deduplicated)} duplicate structure IDs")

# --- Step 2: Merge eligible_df with clinics_df to get incl_ctrl_age ---
eligible_with_age = pd.merge(
    eligible_df,
    clinics_df_deduplicated[['incl_ctrl_bsid', 'incl_ctrl_age']],
    left_on='Bounded Structure ID Number',
    right_on='incl_ctrl_bsid',
    how='left'  # LEFT JOIN preserves all eligible_df rows
)

# Verify row count preserved
print(f"After clinic merge: {len(eligible_with_age)} rows (should be {len(eligible_df)})")

# --- Step 3: Aggregate household_df to one row per bs_id ---
# household_df has 89,663 rows but only 19,891 unique bs_ids
household_agg = household_df.groupby('bs_id', as_index=False).agg({
    'flood_experience': 'first'  # Take first value if multiple
})

print(f"Household data: {len(household_df)} rows → {len(household_agg)} unique structures")

# --- Step 4: Merge flood info - LEFT JOIN preserves all rows ---
eligible_flood_df = pd.merge(
    eligible_with_age,
    household_agg[['bs_id', 'flood_experience']],
    left_on='Bounded Structure ID Number',
    right_on='bs_id',
    how='left'
)

# Verify row count still preserved
print(f"After flood merge: {len(eligible_flood_df)} rows")

# --- Step 5: Fill missing flood info ---
eligible_flood_df['flood_experience'] = eligible_flood_df['flood_experience'].fillna('Unknown')

# --- Step 6: Keep only relevant columns and create copy ---
eligible_full_df = eligible_flood_df[[
    'Record ID',
    'Event Name',
    "Participant's Gender",
    'incl_ctrl_age',
    'Bounded Structure ID Number',
    'flood_experience'
]].copy()

# --- Step 7: Anonymize IDs while preserving relationships ---

# Create anonymous Record IDs (unique per participant)
eligible_full_df['Record ID'] = [f"RID_{i+1:05d}" for i in range(len(eligible_full_df))]

# Create anonymous Structure IDs (same for participants in same structure)
unique_structures = eligible_full_df['Bounded Structure ID Number'].unique()
structure_mapping = {old_id: f"STRUCT_{i+1:04d}" for i, old_id in enumerate(unique_structures)}
eligible_full_df['Bounded Structure ID Number'] = eligible_full_df['Bounded Structure ID Number'].map(structure_mapping)

# --- Step 8: Final summary ---
print("\n" + "="*50)
print("FINAL DATASET SUMMARY")
print("="*50)
print(f"Total participants: {len(eligible_full_df):,}")
print(f"Total unique structures: {eligible_full_df['Bounded Structure ID Number'].nunique():,}")
print(f"\nFlood Experience Distribution:")
print(eligible_full_df['flood_experience'].value_counts().to_string())
print(f"\nAge Data:")
print(f"  - Available: {eligible_full_df['incl_ctrl_age'].notna().sum():,} participants")
print(f"  - Missing: {eligible_full_df['incl_ctrl_age'].isna().sum():,} participants")
print(f"\nGender Distribution:")
print(eligible_full_df["Participant's Gender"].value_counts(dropna=False).to_string())

# --- Step 9: Preview ---
print("\n" + "="*50)
print("DATA PREVIEW (first 10 rows)")
print("="*50)
#eligible_full_df.head(10)






Original clinics rows: 6275
After removing duplicates: 5198
Removed 1077 duplicate structure IDs
After clinic merge: 6295 rows (should be 6295)
Household data: 89663 rows → 19891 unique structures
After flood merge: 6295 rows

FINAL DATASET SUMMARY
Total participants: 6,295
Total unique structures: 5,198

Flood Experience Distribution:
flood_experience
Unknown    5129
No         1051
Yes         115

Age Data:
  - Available: 6,275 participants
  - Missing: 20 participants

Gender Distribution:
Participant's Gender
Female    5131
Male      1144
NaN         20

DATA PREVIEW (first 10 rows)


In [ ]:
len(eligible_full_df)

6295

In [ ]:

# Save to CSV
eligible_full_df.to_csv('eligible_participants_with_flood_data.csv', index=False)